In [1]:
import time
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, BooleanType
from pyspark.sql.functions import col, sha2, current_timestamp, days

print("⏳ Bước 0: Đang khởi tạo Spark và kết nối Iceberg...")
start_time = time.time()

spark = SparkSession.builder \
    .appName("AmazonReviews_Bronze_Full_Ingestion") \
    .config("spark.sql.catalog.gcp_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.gcp_catalog.type", "hadoop") \
    .config("spark.sql.catalog.gcp_catalog.warehouse", "gs://amazon-reviews-lakehouse-warehouse/warehouse/") \
    .getOrCreate()

print("✅ Bước 0 Xong!")

# Định nghĩa Schema (Lược bỏ các cột phức tạp không cần thiết cho phân tích để tối ưu I/O)
amazon_schema = StructType([
    StructField("asin", StringType(), True),
    StructField("helpful_vote", LongType(), True),
    StructField("parent_asin", StringType(), True),
    StructField("rating", DoubleType(), True),
    StructField("text", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("title", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("verified_purchase", BooleanType(), True)
])

raw_data_path = "gs://project-528e2858-1a08-4d22-bcd-lakehouse-bronze/amazon-reviews-raw/*.jsonl"

try:
    print(f"\n⏳ Bước 1 & 2: Đang thiết lập luồng đọc toàn bộ dữ liệu (100GB+) và xử lý ẩn danh PII...")
    df_raw = spark.read.schema(amazon_schema).json(raw_data_path)
    
    df_bronze = df_raw \
        .withColumn("ingest_ts", current_timestamp()) \
        .withColumn("user_id_hashed", sha2(col("user_id"), 256)) \
        .drop("user_id")
    
    print("\n⏳ Bước 3: Đang bắt đầu quá trình GHI TOÀN BỘ dữ liệu xuống Iceberg Bậc Đồng...")
    print("   ⚠️ LƯU Ý: Quá trình này có thể mất từ 15 phút đến 1 tiếng tùy cấu hình máy chủ. Hãy kiên nhẫn!")
    
    # ĐÃ SỬA: Dùng createOrReplace để xóa bảng cũ, ghi đè bảng mới chuẩn xác 100%
    df_bronze.writeTo("gcp_catalog.bronze.amazon_reviews") \
        .tableProperty("write.format.default", "parquet") \
        .tableProperty("write.parquet.compression-codec", "snappy") \
        .partitionedBy(days("ingest_ts")) \
        .createOrReplace() 
        
    print("✅ Bước 3 Xong! Đã đúc thành công toàn bộ dữ liệu vào Bậc Đồng.")
    
    print("\n⏳ Bước 4: Kiểm tra tổng số dòng hiện có trong Bậc Đồng...")
    total_rows = spark.read.table("gcp_catalog.bronze.amazon_reviews").count()
    print(f"🎉 XUẤT SẮC! Tổng số dòng hiện tại trong Lakehouse là: {total_rows} dòng.")

except Exception as e:
    print(f"\n❌ LỖI TRONG QUÁ TRÌNH CHẠY:\n{str(e)}")

end_time = time.time()
print(f"\n⏱️ Tổng thời gian chạy: {round(end_time - start_time, 2)} giây.")

⏳ Bước 0: Đang khởi tạo Spark và kết nối Iceberg...
:: loading settings :: url = jar:file:/usr/lib/spark/jars/ivy-2.5.2.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-fd0b930d-0067-4363-8c47-d82abe39a16b;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.0 in central
:: resolution report :: resolve 167ms :: artifacts dl 4ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-subm

✅ Bước 0 Xong!

⏳ Bước 1 & 2: Đang thiết lập luồng đọc toàn bộ dữ liệu (100GB+) và xử lý ẩn danh PII...



⏳ Bước 3: Đang bắt đầu quá trình GHI TOÀN BỘ dữ liệu xuống Iceberg Bậc Đồng...
   ⚠️ LƯU Ý: Quá trình này có thể mất từ 15 phút đến 1 tiếng tùy cấu hình máy chủ. Hãy kiên nhẫn!


✅ Bước 3 Xong! Đã đúc thành công toàn bộ dữ liệu vào Bậc Đồng.

⏳ Bước 4: Kiểm tra tổng số dòng hiện có trong Bậc Đồng...
🎉 XUẤT SẮC! Tổng số dòng hiện tại trong Lakehouse là: 606938086 dòng.

⏱️ Tổng thời gian chạy: 1848.11 giây.
